# task 1 aug

task-1-gap + own-station context at +-10/15/30/60 min + mask augmentation (real hiding pattern borrowed from a nearby day) + CatBoost GPU blend. A held-out check picks the final model.


In [ ]:
import bisect
import json
import subprocess
import sys
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd

REL = Path("/kaggle/input/competitions/2026-ieee-big-data-traffic-flow-bench/kaggle_public")
OUT = Path("/kaggle/working")
REPO = Path("/tmp/trafficflowbench-public")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/jacky850/trafficflowbench-public.git", str(REPO)], check=True)
print("organizers repo", subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
                                        capture_output=True, text=True).stdout.strip())
sys.path.insert(0, str(REPO / "src"))
from task1 import score_task1 as s1
from task3 import score_task3 as s3

RUN_EVAL = True
PANELS = None
HISTORY_DAYS = 56
GRID_EVERY = 7
ALL_DAYS = False
COMPARE = False
EXPECTED_ROWS = {"validation": 3_451_376, "private": 3_284_419}

KEYS = ["timestamp", "station_id", "link_id"]
VALUES = ["speed_kmh", "flow_vph"]
READ_COLS = KEYS + VALUES + ["pct_observed"]
REGIMES = ["R1", "R2", "R3"]

OLD = dict(max_gap=60, max_span=None, joint=True, cross_day=False)
VARIANTS = {
    "old": OLD,
    "separate": dict(OLD, joint=False),
    "separate+xday": dict(OLD, joint=False, cross_day=True),
    "separate+xday gap45": dict(OLD, joint=False, cross_day=True, max_gap=45),
    "separate+xday gap90": dict(OLD, joint=False, cross_day=True, max_gap=90),
    "separate+xday span90": dict(OLD, joint=False, cross_day=True, max_span=90),
}

panels = PANELS or sorted(p.name for p in (REL / "corridors").iterdir() if p.is_dir())
families = {p["corridor_id"]: p["family_id"] for p in json.loads((REPO / "config" / "corridors.json").read_text())["panels"]}
modes = json.loads((REPO / "config" / "task3_lwr_modes.json").read_text())
print("roads:", panels)

def file_date(path):
    return pd.Timestamp(Path(path).stem[-10:].replace("_", "-"))


@lru_cache(maxsize=4)
def _read(path):
    df = pd.read_parquet(path, columns=READ_COLS)
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df["station_id"] = df["station_id"].astype(str)
    df["link_id"] = df["link_id"].astype(str)
    return df


AUG = {}  # str(path) -> MultiIndex of (station_id, timestamp) readings to hide (mask augmentation)


def read_day(path):
    df = _read(path).copy()
    drop = AUG.get(str(path))
    if drop is not None:
        hit = pd.MultiIndex.from_arrays([df["station_id"], df["timestamp"]]).isin(drop)
        df.loc[hit, ["speed_kmh", "flow_vph"]] = np.nan
    return df


def masked_days(panel, split):
    days = {}
    for regime in REGIMES:
        for f in sorted((REL / "corridors" / panel / split / "mainline_states_masked" / f"mask_regime={regime}").glob("*.parquet")):
            assert file_date(f) not in days, f"{f.name} in two regimes"
            days[file_date(f)] = (regime, f)
    return dict(sorted(days.items()))


def daily_profiles(panel):
    aggs = {}
    for f in sorted((REL / "corridors" / panel / "train" / "mainline_states").rglob("*.parquet")):
        d = pd.read_parquet(f, columns=READ_COLS)
        d["timestamp"] = pd.to_datetime(d["timestamp"], utc=True)
        d["link_id"] = d["link_id"].astype(str)
        ok = d["pct_observed"] >= 75
        d["sv"] = d["speed_kmh"].where(ok)
        d["fv"] = d["flow_vph"].where(ok)
        d["weekday"] = d["timestamp"].dt.dayofweek
        d["hour"] = d["timestamp"].dt.hour
        aggs[file_date(f)] = d.groupby(["link_id", "weekday", "hour"]).agg(
            speed_sum=("sv", "sum"), speed_n=("sv", "count"), flow_sum=("fv", "sum"), flow_n=("fv", "count"))
    return aggs


def hist_before(aggs, date):
    dates = sorted(aggs)
    used = dates[:bisect.bisect_left(dates, date)][-HISTORY_DAYS:]
    if not used:
        return None
    return pd.concat([aggs[d] for d in used]).groupby(level=[0, 1, 2]).sum()


def profile_values(rows, hist):
    n = len(rows)
    if hist is None:
        return np.full(n, np.nan), np.full(n, np.nan)
    h = hist.reset_index()
    keyed = pd.DataFrame({"link_id": rows["link_id"].to_numpy(),
                          "weekday": rows["timestamp"].dt.dayofweek.to_numpy(),
                          "hour": rows["timestamp"].dt.hour.to_numpy()})
    out = []
    for v in ("speed", "flow"):
        by_day = h.assign(m=h[f"{v}_sum"] / h[f"{v}_n"].replace(0, np.nan))[["link_id", "weekday", "hour", "m"]]
        by_hour = h.groupby(["link_id", "hour"])[[f"{v}_sum", f"{v}_n"]].sum()
        by_hour = (by_hour[f"{v}_sum"] / by_hour[f"{v}_n"].replace(0, np.nan)).rename("f").reset_index()
        overall = h[f"{v}_sum"].sum() / max(h[f"{v}_n"].sum(), 1)
        k = keyed.merge(by_day, on=["link_id", "weekday", "hour"], how="left").merge(by_hour, on=["link_id", "hour"], how="left")
        out.append(k["m"].fillna(k["f"]).fillna(overall).to_numpy())
    return out


def straight_line(pv, nv, dp, dn, cfg):
    pv, nv, dp, dn = (np.asarray(a, dtype=float) for a in (pv, nv, dp, dn))
    prev_ok = ~np.isnan(pv) & (dp <= cfg["max_gap"])
    next_ok = ~np.isnan(nv) & (dn <= cfg["max_gap"])
    both = prev_ok & next_ok
    if cfg["max_span"]:
        both &= (dp + dn) <= cfg["max_span"]
    res = np.full(len(pv), np.nan)
    res[prev_ok] = pv[prev_ok]
    res[next_ok & ~prev_ok] = nv[next_ok & ~prev_ok]
    nearer_next = prev_ok & next_ok & ~both & (dn < dp)
    res[nearer_next] = nv[nearer_next]
    res[both] = (pv[both] * dn[both] + nv[both] * dp[both]) / (dp[both] + dn[both])
    return res


def predict_day(day_files, date, targets, hist, cfg):
    dates = [date - pd.Timedelta(days=1), date, date + pd.Timedelta(days=1)] if cfg["cross_day"] else [date]
    df = pd.concat([read_day(day_files[d][1]) for d in dates if d in day_files], ignore_index=True)
    df = df.sort_values(["link_id", "station_id", "timestamp"])
    keys = [df["link_id"], df["station_id"]]
    ok = df["pct_observed"] >= 75
    have = {c: df[c].notna() for c in VALUES}
    both_present = have["speed_kmh"] & have["flow_vph"]
    ctx = df[KEYS].copy()
    for c in VALUES:
        good = ok & (both_present if cfg["joint"] else have[c])
        seen, vals = df["timestamp"].where(good), df[c].where(good)
        ctx["pt_" + c] = seen.groupby(keys).ffill()
        ctx["nt_" + c] = seen.groupby(keys).bfill()
        ctx["pv_" + c] = vals.groupby(keys).ffill()
        ctx["nv_" + c] = vals.groupby(keys).bfill()
    out = targets.merge(ctx, on=KEYS, how="left", validate="one_to_one")
    h_speed, h_flow = profile_values(out, hist)
    result, gap = {}, {}
    for c, h in (("speed_kmh", h_speed), ("flow_vph", h_flow)):
        dp = (out["timestamp"] - out["pt_" + c]).dt.total_seconds().to_numpy() / 60
        dn = (out["nt_" + c] - out["timestamp"]).dt.total_seconds().to_numpy() / 60
        line = straight_line(out["pv_" + c], out["nv_" + c], dp, dn, cfg)
        good = ok & have[c]
        link_mean = out["link_id"].map(df[good].groupby("link_id")[c].mean()).to_numpy()
        filled = pd.Series(line).fillna(pd.Series(h)).fillna(pd.Series(link_mean)).fillna(df.loc[good, c].mean())
        result[c] = filled.to_numpy()
        if c == "speed_kmh":
            result["span_min"] = np.nan_to_num(dp, nan=999) + np.nan_to_num(dn, nan=999)
        k = c[0]  # gap edges: minutes to the last/next good reading, their values and times
        gap[f"g_dp_{k}"], gap[f"g_dn_{k}"] = dp.astype(np.float32), dn.astype(np.float32)
        gap[f"g_pv_{k}"] = out["pv_" + c].to_numpy(np.float32); gap[f"g_nv_{k}"] = out["nv_" + c].to_numpy(np.float32)
        gap[f"g_pt_{k}"] = out["pt_" + c].values; gap[f"g_nt_{k}"] = out["nt_" + c].values  # naive UTC datetime64
    return np.clip(result["speed_kmh"], 1, None), np.clip(result["flow_vph"], 0, None), result["span_min"], gap


def load_sheet(panel, split, dates=None):
    path = REL / "task1" / panel / split / "sample_submission_state.csv"
    parts = []
    for chunk in pd.read_csv(path, dtype=str, chunksize=500_000):
        if dates is not None:
            chunk = chunk[chunk["timestamp"].str[:10].isin(dates)]
        if len(chunk):
            parts.append(chunk)
    sheet = pd.concat(parts, ignore_index=True)
    sheet["ts"] = pd.to_datetime(sheet["timestamp"], utc=True)
    sheet["date"] = sheet["timestamp"].str[:10]
    return sheet


def predict_split(panel, split, cfg, aggs, only_dates=None, hist_fixed=None):
    days = masked_days(panel, split)
    if only_dates is not None:
        days_used = {d: v for d, v in days.items() if d in only_dates}
    else:
        days_used = days
    sheet = load_sheet(panel, split, {d.strftime("%Y-%m-%d") for d in days_used} if only_dates is not None else None)
    pieces = []
    for date, (regime, _) in days_used.items():
        rows = sheet[(sheet["date"] == date.strftime("%Y-%m-%d")) & (sheet["mask_regime"] == regime)]
        if rows.empty:
            continue
        targets = rows[["station_id", "link_id"]].copy()
        targets["timestamp"] = rows["ts"].to_numpy()
        hist = hist_fixed if hist_fixed is not None else hist_before(aggs, date)
        speed, flow, span, gap = predict_day(days, date, targets, hist, cfg)
        piece = rows[["panel", "timestamp", "ts", "station_id", "link_id", "mask_regime"]].copy()
        piece["speed_kmh"], piece["flow_vph"], piece["span_min"] = speed, flow, span
        for k, v in gap.items():
            piece[k] = v
        pieces.append(piece)
    return pd.concat(pieces, ignore_index=True), sheet

def official_task1(panel, preds, names=None):
    original = s1.masked_files
    if names is not None:
        s1.masked_files = lambda rel, p, split, regime: [f for f in original(rel, p, split, regime) if f.name in names]
    try:
        sub = preds.drop(columns=["timestamp"]).rename(columns={"ts": "timestamp"})
        return pd.DataFrame(s1.score_panel(panel, REL, "train", sub))
    finally:
        s1.masked_files = original


def overall(detail):
    p = detail[detail["level"] == "panel"].assign(family=lambda d: d["panel"].map(families))
    return p.groupby("family")["S_state"].mean().mean()


def sample_days(days):
    return {d for r in REGIMES for d in [d for d, (rr, _) in days.items() if rr == r][::GRID_EVERY]}


def official_fd(panel, preds, names=None):
    sub = preds.drop(columns=["timestamp"]).rename(columns={"ts": "timestamp"})
    original = s3.files
    if names is not None:
        s3.files = lambda panel_dir, split: [f for f in original(panel_dir, split) if f.name in names]
    try:
        rows = []
        for r in REGIMES:
            d, missing = s3.derive_physics(panel, REL, "train", sub, r)
            res = s3.score_panel(panel, d, REL, modes, r)
            rows.append({"panel": panel, "regime": r, "S_FD": res.get("S_FD", 0.0), "missing_targets": missing})
        return pd.DataFrame(rows)
    finally:
        s3.files = original


def error_breakdown(panel, preds):
    truth_files = {f.name: f for f in (REL / "corridors" / panel / "train" / "mainline_states").rglob("*.parquet")}
    days = masked_days(panel, "train")
    lanes_map = s1.station_lanes(REL / "corridors" / panel)
    topo = pd.read_csv(REL / "corridors" / panel / "network" / "lwr_mainline_topology.csv", dtype={"link_id": str})
    free = topo.drop_duplicates("link_id").set_index("link_id")["free_speed_kmh"]
    preds = preds.assign(day=preds["ts"].dt.normalize().dt.tz_localize(None))
    parts = []
    for date, (regime, f) in days.items():
        p = preds[preds["day"] == date]
        if p.empty:
            continue
        t = read_day(truth_files[f.name])[KEYS + VALUES].rename(columns={"speed_kmh": "s_true", "flow_vph": "f_true"})
        m = p.drop(columns=["timestamp"]).rename(columns={"ts": "timestamp"}).merge(t, on=KEYS, how="inner")
        parts.append(m)
    m = pd.concat(parts, ignore_index=True).dropna(subset=["s_true", "f_true"])
    lanes = s1.lane_vector(m["station_id"], m["link_id"], lanes_map)
    m["se_speed"] = (m["speed_kmh"] - m["s_true"]) ** 2
    m["se_flow_lane"] = ((m["flow_vph"] - m["f_true"]) / lanes) ** 2
    m["jam"] = m["s_true"] <= 0.6 * m["link_id"].map(free)
    m["gap"] = pd.cut(m["span_min"], [0, 15, 30, 60, 120, 1e9], labels=["<=15", "15-30", "30-60", "60-120", ">120 or one side"])
    g = m.groupby(["jam", "gap"], observed=True).agg(cells=("se_speed", "size"), mse_speed=("se_speed", "mean"),
                                                     mse_flow_lane=("se_flow_lane", "mean"))
    g["rmse_speed"], g["rmse_flow_lane"] = np.sqrt(g.pop("mse_speed")), np.sqrt(g.pop("mse_flow_lane"))
    return g.reset_index().assign(panel=panel)

In [ ]:
import time
import warnings
import lightgbm as lgb
from catboost import CatBoostRegressor
warnings.filterwarnings('ignore')
PARAMS = {'objective': 'regression', 'learning_rate': 0.05, 'num_leaves': 63, 'min_data_in_leaf': 200, 'feature_fraction': 0.8,
          'bagging_fraction': 0.8, 'bagging_freq': 1, 'lambda_l2': 1.0, 'num_threads': 4, 'verbose': -1, 'seed': 2026}
ROUNDS = 400
CB_ITERS_CHECK, CB_ITERS_FINAL = 1500, 3000
CHECK_ROWS = 3_000_000
AUG_ON = True
FE = ['speed_kmh', 'flow_vph', 'span_min', 'f_u1', 'f_d1', 'f_u2', 'f_d2', 's_u1', 's_d1', 's_u2', 's_d2', 'nb_flow', 'nb_speed',
      'flow_minus_nb', 'speed_minus_nb', 'hour', 'dow', 'lanes', 'free', 'regime']
GAP_FE = [f'g_{a}_{k}' for k in 'sf' for a in ('dp', 'dn', 'pv', 'nv')] + \
         [f'{a}_{k}_{e}' for k in 'sf' for a in ('nbd', 'est') for e in 'pn']
CTX_OFF = (-12, -6, -3, -2, 2, 3, 6, 12)  # own station at -60/-30/-15/-10 and +10/+15/+30/+60 min
CTX_FE = [f'own_{k}_{"m" if o < 0 else "p"}{abs(o) * 5}' for o in CTX_OFF for k in 'sf']
FEATS_OLD = FE + ['panel_code']
FEATS_GAP = FEATS_OLD + GAP_FE          # = the 0.84862 task-1-gap model
FEATS_ALL = FEATS_GAP + CTX_FE

def station_order(panel):
    f = sorted((REL / 'corridors' / panel / 'train' / 'mainline_states').rglob('*.parquet'))[0]
    d = pd.read_parquet(f, columns=['station_id', 'milepost']).drop_duplicates('station_id')
    d['station_id'] = d['station_id'].astype(str)
    return d.sort_values('milepost')['station_id'].tolist()

def free_map(panel):
    topo = pd.read_csv(REL / 'corridors' / panel / 'network' / 'lwr_mainline_topology.csv', dtype={'link_id': str})
    return topo.drop_duplicates('link_id').set_index('link_id')['free_speed_kmh']

def add_features(panel, split, preds, truth=False):
    days = masked_days(panel, split)
    truth_files = {f.name: f for f in (REL / 'corridors' / panel / 'train' / 'mainline_states').rglob('*.parquet')} if truth else {}
    order = station_order(panel); pos = {s: i for i, s in enumerate(order)}; nS = len(order)
    lanes_map = s1.station_lanes(REL / 'corridors' / panel); free = free_map(panel)
    preds = preds.assign(day=preds['ts'].dt.normalize().dt.tz_localize(None))
    out = []
    for date, (regime, f) in days.items():
        p = preds[preds['day'] == date].copy()
        if p.empty:
            continue
        df = read_day(f); df = df[df['pct_observed'] >= 75]
        grid = pd.date_range(pd.Timestamp(date).tz_localize('UTC'), periods=288, freq='5min')  # full day, so row offsets = minutes
        F = df.pivot_table(index='timestamp', columns='station_id', values='flow_vph', aggfunc='mean').reindex(index=grid, columns=order)
        S = df.pivot_table(index='timestamp', columns='station_id', values='speed_kmh', aggfunc='mean').reindex(index=grid, columns=order)
        Fv, Sv = F.to_numpy(), S.to_numpy()
        ti = F.index.get_indexer(p['ts']); si = p['station_id'].map(pos).fillna(-100).astype(int).to_numpy()
        def take(M, k):
            j = si + k
            okb = (j >= 0) & (j < nS) & (ti >= 0)
            res = np.full(len(p), np.nan)
            res[okb] = M[ti[okb], j[okb]]
            return res
        for k, name in ((-1, 'u1'), (1, 'd1'), (-2, 'u2'), (2, 'd2')):
            p['f_' + name] = take(Fv, k); p['s_' + name] = take(Sv, k)
        near_f = np.nanmean(np.vstack([p['f_u1'], p['f_d1']]), axis=0)
        far_f = np.nanmean(np.vstack([p['f_u2'], p['f_d2']]), axis=0)
        near_s = np.nanmean(np.vstack([p['s_u1'], p['s_d1']]), axis=0)
        far_s = np.nanmean(np.vstack([p['s_u2'], p['s_d2']]), axis=0)
        p['nb_flow'] = np.where(np.isnan(near_f), far_f, near_f); p['nb_speed'] = np.where(np.isnan(near_s), far_s, near_s)
        p['flow_minus_nb'] = p['flow_vph'] - p['nb_flow']; p['speed_minus_nb'] = p['speed_kmh'] - p['nb_speed']
        # neighbours at the gap edges: how much they changed since the last good reading / until the next one
        def nb_at(M, tix):
            res = []
            for ks in ((-1, 1), (-2, 2)):
                vals = []
                for k in ks:
                    j = si + k; okb = (j >= 0) & (j < nS) & (tix >= 0)
                    r = np.full(len(p), np.nan); r[okb] = M[tix[okb], j[okb]]; vals.append(r)
                res.append(np.nanmean(np.vstack(vals), axis=0))
            return np.where(np.isnan(res[0]), res[1], res[0])
        for k, M, nb in (('f', Fv, p['nb_flow'].to_numpy()), ('s', Sv, p['nb_speed'].to_numpy())):
            for e, col in (('p', f'g_pt_{k}'), ('n', f'g_nt_{k}')):
                tix = F.index.get_indexer(pd.DatetimeIndex(p[col].to_numpy()).tz_localize('UTC'))
                d = nb - nb_at(M, tix)
                p[f'nbd_{k}_{e}'] = d
                p[f'est_{k}_{e}'] = p[f'g_{"pv" if e == "p" else "nv"}_{k}'].to_numpy() + d
        okS = (si >= 0) & (si < nS) & (ti >= 0)
        for o in CTX_OFF:  # the station's own released readings around the target time
            r = ti + o; okr = okS & (r >= 0) & (r < len(grid))
            for k, M in (('s', Sv), ('f', Fv)):
                v = np.full(len(p), np.nan); v[okr] = M[r[okr], si[okr]]
                p[f'own_{k}_{"m" if o < 0 else "p"}{abs(o) * 5}'] = v
        m = p.rename(columns={'timestamp': 'ts_str', 'ts': 'timestamp'})
        if truth:
            t = read_day(truth_files[f.name])[KEYS + VALUES].rename(columns={'speed_kmh': 's_true', 'flow_vph': 'f_true'})
            m = m.merge(t, on=KEYS, how='left')
        m['hour'] = m['timestamp'].dt.hour; m['dow'] = m['timestamp'].dt.dayofweek
        m['lanes'] = s1.lane_vector(m['station_id'], m['link_id'], lanes_map)
        m['free'] = m['link_id'].map(free).to_numpy(); m['regime'] = REGIMES.index(regime)
        out.append(m)
    return pd.concat(out, ignore_index=True)

# ------------------------------------------------------------------ mask augmentation
# Each sampled training day borrows the organizers' real hiding pattern from a nearby day of the same regime.
# Readings that are released on this day but hidden on the donor day become extra targets: they are blanked
# (read_day honours AUG) before the straight line and the features are built, and their truth comes from the
# unmasked file. Only fit days are augmented; held-out days keep their official targets only.
def donor_of(d, days):
    reg = days[d][0]
    for off in (1, -1, 2, -2, 3, -3, 7, -7):
        c = d + pd.Timedelta(days=off)
        if c in days and days[c][0] == reg:
            return c
    return None

def aug_rows(panel, days, dates, aggs):
    donors = {d: donor_of(d, days) for d in dates}
    need = {x.strftime('%Y-%m-%d') for x in donors.values() if x is not None}
    if not need:
        return None
    sheet = load_sheet(panel, 'train', need)
    out = []
    for d, dn in donors.items():
        if dn is None:
            continue
        regime, f = days[d]
        r = sheet[sheet['date'] == dn.strftime('%Y-%m-%d')]
        ts = pd.DatetimeIndex(r['ts']) + (d - dn)
        raw = _read(f)
        ok = (raw['pct_observed'] >= 75) & raw['speed_kmh'].notna() & raw['flow_vph'].notna()
        obs = pd.MultiIndex.from_arrays([raw.loc[ok, 'station_id'].to_numpy(), pd.DatetimeIndex(raw.loc[ok, 'timestamp'])])
        cand = pd.MultiIndex.from_arrays([r['station_id'].to_numpy(), ts])
        keep = cand.isin(obs)
        if not keep.any():
            continue
        tgt = pd.DataFrame({'station_id': r['station_id'].to_numpy()[keep], 'link_id': r['link_id'].to_numpy()[keep], 'timestamp': ts[keep]})
        AUG[str(f)] = cand[keep]
        try:
            speed, flow, span, gap = predict_day(days, d, tgt, hist_before(aggs, d), OLD)
            piece = pd.DataFrame({'panel': panel, 'timestamp': tgt['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ'), 'ts': tgt['timestamp'],
                                  'station_id': tgt['station_id'], 'link_id': tgt['link_id'], 'mask_regime': regime})
            piece['speed_kmh'], piece['flow_vph'], piece['span_min'] = speed, flow, span
            for k_, v_ in gap.items():
                piece[k_] = v_
            out.append(add_features(panel, 'train', piece, truth=True))
        finally:
            AUG.pop(str(f), None)
    return pd.concat(out, ignore_index=True) if out else None

COLS = FEATS_ALL + ['s_true', 'f_true']
AGGS, TRAIN = {}, []
for panel in panels:
    ts0 = time.time()
    AGGS[panel] = daily_profiles(panel)
    days = masked_days(panel, 'train')
    dates = sorted(sample_days(days))
    hold_days = set(pd.Timestamp(d) for d in dates[2::4])  # every 4th sampled day, never used to fit the check models
    preds, _ = predict_split(panel, 'train', OLD, AGGS[panel], only_dates=set(dates))
    t = add_features(panel, 'train', preds, truth=True)
    t['panel_code'] = panels.index(panel)
    t['_hold'] = t['day'].isin(hold_days).astype('float32'); t['_aug'] = np.float32(0)
    TRAIN.append(t[COLS + ['_hold', '_aug']].astype({c: 'float32' for c in COLS}))
    n_aug = 0
    if AUG_ON:
        ta = aug_rows(panel, days, [d for d in dates if pd.Timestamp(d) not in hold_days], AGGS[panel])
        if ta is not None:
            ta['panel_code'] = panels.index(panel); ta['_hold'] = np.float32(0); ta['_aug'] = np.float32(1)
            TRAIN.append(ta[COLS + ['_hold', '_aug']].astype({c: 'float32' for c in COLS})); n_aug = len(ta)
            del ta
    print('train rows', panel, len(t), '+ augmented', n_aug, 'sec', round(time.time() - ts0), flush=True)
    del preds, t
TR = pd.concat(TRAIN, ignore_index=True); del TRAIN
TR = TR[TR['s_true'].notna() & TR['f_true'].notna()].reset_index(drop=True)
print('training rows', len(TR), '| augmented share', round(float(TR['_aug'].mean()), 3), flush=True)

# ------------------------------------------------------------------ models
def cb_fit(X, y, iters):
    kw = dict(iterations=iters, depth=8, learning_rate=0.08, loss_function='RMSE', random_seed=2026, verbose=0)
    try:
        m = CatBoostRegressor(task_type='GPU', border_count=128, **kw); m.fit(X, y)
    except Exception as exc:  # no GPU in this session: CPU fallback, slower
        print('CatBoost GPU unavailable, using CPU:', repr(exc)[:200], flush=True)
        m = CatBoostRegressor(thread_count=4, **kw); m.fit(X, y)
    return m

def fit_models(kind, cols, D, cb_iters):
    X = D[cols].to_numpy(np.float32)
    rf, rs = (D['f_true'] - D['flow_vph']).to_numpy(np.float32), (D['s_true'] - D['speed_kmh']).to_numpy(np.float32)
    M = {'cols': cols}
    if kind in ('lgb', 'blend'):
        M['lgb'] = (lgb.train(PARAMS, lgb.Dataset(X, rf), ROUNDS), lgb.train(PARAMS, lgb.Dataset(X, rs), ROUNDS))
    if kind in ('cb', 'blend'):
        M['cb'] = (cb_fit(X, rf, cb_iters), cb_fit(X, rs, cb_iters))
    return M

def residuals(M, D):
    X = D[M['cols']].to_numpy(np.float32); outs = []
    for key in ('lgb', 'cb'):
        if key in M:
            outs.append((M[key][0].predict(X), M[key][1].predict(X)))
    return np.mean([o[0] for o in outs], axis=0), np.mean([o[1] for o in outs], axis=0)

# ------------------------------------------------------------------ check on held-out days (official targets only)
t1 = time.time()
hold = TR['_hold'].to_numpy() > 0; aug = TR['_aug'].to_numpy() > 0
rng = np.random.default_rng(0)
pick = lambda m: rng.choice(np.flatnonzero(m), min(int(m.sum()), CHECK_ROWS), replace=False)
off_i, all_i = pick(~hold & ~aug), pick(~hold)
H = TR[hold]
def rmse(rf, rs):
    return (float(np.sqrt(np.mean((H['speed_kmh'] + rs - H['s_true']) ** 2))),
            float(np.sqrt(np.mean((((H['flow_vph'] + rf).clip(lower=0) - H['f_true']) / H['lanes'].clip(lower=1)) ** 2))))
CHECK = {'straight line': rmse(0, 0)}
CONF = {'gap (0.84862)': ('lgb', FEATS_GAP, off_i), '+ctx': ('lgb', FEATS_ALL, off_i),
        '+ctx +aug': ('lgb', FEATS_ALL, all_i), 'catboost +ctx +aug': ('cb', FEATS_ALL, all_i)}
RES = {}
for name, (kind, cols, idx) in CONF.items():
    M = fit_models(kind, cols, TR.iloc[idx], CB_ITERS_CHECK)
    RES[name] = residuals(M, H); CHECK[name] = rmse(*RES[name]); del M
    print('check', name, CHECK[name], round(time.time() - t1), 's', flush=True)
RES['blend lgb+cb +ctx +aug'] = tuple((a + b) / 2 for a, b in zip(RES['+ctx +aug'], RES['catboost +ctx +aug']))
CHECK['blend lgb+cb +ctx +aug'] = rmse(*RES['blend lgb+cb +ctx +aug'])
print(pd.DataFrame(CHECK, index=['rmse speed km/h', 'rmse flow per lane']).round(3).to_string(), flush=True)
g = CHECK['gap (0.84862)']
rel = {k: (v[0] / g[0] + v[1] / g[1]) / 2 for k, v in CHECK.items() if k != 'straight line'}
CHOICE = min(rel, key=rel.get)
FINAL = {'gap (0.84862)': ('lgb', FEATS_GAP, False), '+ctx': ('lgb', FEATS_ALL, False), '+ctx +aug': ('lgb', FEATS_ALL, True),
         'catboost +ctx +aug': ('cb', FEATS_ALL, True), 'blend lgb+cb +ctx +aug': ('blend', FEATS_ALL, True)}[CHOICE]
print('relative error vs gap model:', {k: round(v, 4) for k, v in rel.items()}, '-> using', CHOICE, flush=True)
del H, RES

t1 = time.time()
kind, FEATS, use_aug = FINAL
MODEL = fit_models(kind, FEATS, TR if use_aug else TR[~aug], CB_ITERS_FINAL)
print('final', kind, 'trained on', len(TR) if use_aug else int((~aug).sum()), 'rows in', round(time.time() - t1), 'sec', flush=True)
del TR

def build_split(split):
    out_path = OUT / f'state_{split}.csv'
    total, first = 0, True
    for panel in panels:
        aggs = AGGS[panel]
        first_day = min(masked_days(panel, split))
        result, sheet = predict_split(panel, split, OLD, aggs, hist_fixed=hist_before(aggs, first_day))
        feats = add_features(panel, split, result)
        assert len(feats) == len(result), (panel, split, len(feats), len(result))
        feats['panel_code'] = panels.index(panel)
        rf, rs = residuals(MODEL, feats)
        flow = np.clip(feats['flow_vph'] + rf, 0, None)
        speed = np.clip(feats['speed_kmh'] + rs, 1, None)
        k5 = ['panel', 'timestamp', 'station_id', 'link_id', 'mask_regime']
        final = feats[['panel', 'ts_str', 'station_id', 'link_id', 'mask_regime']].rename(columns={'ts_str': 'timestamp'}).assign(speed_kmh=speed.to_numpy(), flow_vph=flow.to_numpy())
        assert len(final) == len(sheet), (panel, split, len(final), len(sheet))
        assert not final.duplicated(k5).any(), (panel, split)
        assert final.merge(sheet[k5], on=k5, how='outer', indicator=True)['_merge'].eq('both').all(), (panel, split)
        assert np.isfinite(final[VALUES].to_numpy()).all(), (panel, split)
        final[k5 + VALUES].to_csv(out_path, mode='w' if first else 'a', header=first, index=False)
        first = False
        total += len(final)
        print(split, panel, len(final), 'mean change speed', round(float((speed.to_numpy() - feats['speed_kmh'].to_numpy()).mean()), 3),
              'flow', round(float((flow.to_numpy() - feats['flow_vph'].to_numpy()).mean()), 2), flush=True)
    assert total == EXPECTED_ROWS[split], (split, total)
    return out_path

parts = [build_split(s) for s in ['validation', 'private']]
with open(OUT / 'state.csv', 'w') as out:
    for i, p in enumerate(parts):
        with open(p) as f:
            header = f.readline()
            if i == 0:
                out.write(header)
            out.writelines(f)
        p.unlink()
n = sum(1 for _ in open(OUT / 'state.csv')) - 1
print('state.csv rows:', f'{n:,}')
assert n == sum(EXPECTED_ROWS.values()), n
